In [1]:
# 필수 라이브러리 설치
%pip install -q diffusers transformers accelerate peft bitsandbytes
%pip install xformers==0.0.27.post2


[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import torch
from diffusers import StableDiffusionXLPipeline

# 모델 로드
pipe = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    dtype=torch.float16,
    # use_safetensors=True,
    variant="fp16"
)
pipe.to("cuda")

# 메모리 최적화
pipe.enable_xformers_memory_efficient_attention()

# 이미지 생성
prompt = "A majestic lion in a sunset savanna, highly detailed, 4k"
negative_prompt = "blurry, low quality, distorted"

image = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    num_inference_steps=30,
    guidance_scale=7.5,
    width=1024,
    height=1024
).images[0]

image.save("sdxl_output.png")

/usr/local/lib/python3.11/dist-packages/xformers/ops/fmha/flash.py:211: FutureWarning: `torch.library.impl_abstract` was renamed to `torch.library.register_fake`. Please use that instead; we will remove `torch.library.impl_abstract` in a future version of PyTorch.
  @torch.library.impl_abstract("xformers_flash::flash_fwd")
/usr/local/lib/python3.11/dist-packages/xformers/ops/fmha/flash.py:344: FutureWarning: `torch.library.impl_abstract` was renamed to `torch.library.register_fake`. Please use that instead; we will remove `torch.library.impl_abstract` in a future version of PyTorch.
  @torch.library.impl_abstract("xformers_flash::flash_bwd")
Keyword arguments {'dtype': torch.float16} are not expected by StableDiffusionXLPipeline and will be ignored.


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

In [7]:
# LoRA 파인튜닝(Diffusers + PEFT(LoRA)
# Stable Diffusion XL  모델의 일부 파라메터만 저비용으로 미세조정 하는 설정 
# 대규모 가중치는 고정
# Attention 계층에만 LoRA 어뎁터를 삽입
# 스타일, 캐릭터, 특정 도메인(의료,산업 이미지)특화된 학습 가능
from diffusers import StableDiffusionXLPipeline, AutoencoderKL
from peft import LoraConfig, get_peft_model
import torch

# LoRA 설정
lora_config = LoraConfig(
    r = 8,   # lora rank ( 16~32  스타일/캐릭터 학습에 유리 vram 증가)
    lora_alpha=32,   # 스케일링 벡터
    target_modules=[
        'to_q','to_k','to_v', 'to_out.0',  # Attention Layers
        'proj_in','proj_out',   # u-net
        'ff.net.0.proj', 'ff.net.2',  # Feed Forward Network
    ],
    lora_dropout = 0.05,
)
# 학습설정(sample)
training_args = {
    'learning_rate' : 1e-4,
    'train_batch_size' : 1,
    'gradient_accumulation_step' : 4,
    'max_train_steps' : 1000,
    'mixed_precision' : 'fp16'
}

In [10]:
# QLoRA 파인튜닝(메모리 효율적)  -4bit 양자화
from transformers import BitsAndBytesConfig
from diffusers import StableDiffusionXLPipeline
from peft import LoraConfig, prepare_model_for_kbit_training
# 4bit 양자화 설정
bnb_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)
# QLoRA 설정
qlora_config = LoraConfig(
    r = 4,   # 더 작은 rank 사용 가능
    lora_alpha=16,   # 스케일링 벡터
    target_modules=[
        'to_q','to_k','to_v', 'to_out.0',  # Attention Layers        
    ],
    lora_dropout = 0.05,
)